In [1]:
import torch
import torch.nn as nn

In [2]:
class ConvBNLeakyReLU(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1):
        super().__init__()
        padding = 1 if kernel_size == 3 else 0
        self.convbnleakyrelu=nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(negative_slope=0.1, inplace=True),
        )

    def forward(self, x):
        return self.convbnleakyrelu(x)

In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.residual = nn.Sequential(
            ConvBNLeakyReLU(channels, channels // 2, 1, 1),
            ConvBNLeakyReLU(channels // 2, channels, 3, 1),
        )

    def forward(self, x):
        return x + self.residual(x)

In [ ]:
class YOLOv3(nn.Module):
    def _make_stage(self, channels, num_blocks):
        blocks_list = []
        for _ in range(num_blocks):
            blocks_list.append(ResidualBlock(channels))
        return nn.Sequential(*blocks_list)

    def _make_head(self, in_channels, channels):
        blocks_list = []
        initial_layer = ConvBNLeakyReLU(in_channels, channels, 1, 1)
        blocks_list.append(initial_layer)
        for _ in range(2):
            next_layers = nn.Sequential(
                ConvBNLeakyReLU(channels, 2 * channels, 3, 1),
                ConvBNLeakyReLU(2 * channels, channels, 1, 1),
            )
            blocks_list.append(next_layers)
        return nn.Sequential(*blocks_list)

    def __init__(self, num_classes):
        super().__init__()
        self.conv_initial = ConvBNLeakyReLU(3, 32, 3, 1)
        self.down1 = ConvBNLeakyReLU(32, 64, 3, 2)
        self.stage1 = self._make_stage(64, 1)
        self.down2 = ConvBNLeakyReLU(64, 128, 3, 2)
        self.stage2 = self._make_stage(128, 2)
        self.down3 = ConvBNLeakyReLU(128, 256, 3, 2)
        self.stage3 = self._make_stage(256, 8)
        self.down4 = ConvBNLeakyReLU(256, 512, 3, 2)
        self.stage4 = self._make_stage(512, 8)
        self.down5 = ConvBNLeakyReLU(512, 1024, 3, 2)
        self.stage5 = self._make_stage(1024, 4)
        self.head13 = self._make_head(1024, 512)
        self.head13_conv6 = ConvBNLeakyReLU(512, 1024, 3, 1)
        self.pred13 = nn.Conv2d(1024, 3 * (5 + num_classes), 1, 1)
        self.reduce13_to26 = ConvBNLeakyReLU(512, 256, 1, 1)
        self.upsample =nn.Upsample(scale_factor=2, mode='nearest')
        self.head26 = self._make_head(768, 256)
        self.head26_conv6 = ConvBNLeakyReLU(256, 512, 3, 1)
        self.pred26 = nn.Conv2d(512, 3 * (5 + num_classes), 1, 1)
        self.reduce26_to52= ConvBNLeakyReLU(256, 128, 1, 1)
        self.head52 = self._make_head(384, 128)
        self.head52_conv6 = ConvBNLeakyReLU(128, 256, 3, 1)
        self.pred52 = nn.Conv2d(256, 3 * (5 + num_classes), 1, 1)
    def forward(self, x):
        x = self.conv_initial(x)
        x = self.stage1(self.down1(x))
        x = self.stage2(self.down2(x))
        x = self.stage3(self.down3(x))
        route_52 = x
        x = self.stage4(self.down4(x))
        route_26 = x
        x = self.stage5(self.down5(x))
        x = self.head13(x)
        route_13 = x
        x = self.head13_conv6(x)
        prediction13 = self.pred13(x)
        route_13 = self.upsample(self.reduce13_to26(route_13))
        x = torch.cat([route_13, route_26], 1)
        x = self.head26(x)
        head_route26 = x
        x = self.head26_conv6(x)
        prediction26 = self.pred26(x)
        head_route26 = self.upsample(self.reduce26_to52(head_route26))
        x = torch.cat([route_52, head_route26], 1)
        x = self.head52(x)
        x = self.head52_conv6(x)
        prediction52 = self.pred52(x)
        return prediction13, prediction26, prediction52